# Semantic-aware HNSW Variant C

This notebook tests whether compiled semantic predicates can influence **HNSW traversal itself** rather than being applied only after ANN retrieval.

It compares on the same graph and queries:

1. **HNSW post-filter** — retrieve an oversampled dense set, then apply the semantic gate.
2. **HNSW filtered traversal** — the library's filtered search; invalid nodes remain traversable but cannot be returned.
3. **Semantic HNSW C** — layer-0 expansion priority is `dense + lambda * semantic`; failed semantic nodes are treated as bridges and can trigger bounded 2-hop expansion.

Ground truth is brute-force top-K dense neighbors among items satisfying the same semantic predicate, so we report **Recall@K and latency**, not speed alone.

**Important current scope:** the prototype precomputes each item's compiled semantic score once before the timed query loop. Therefore the timing in this notebook isolates graph traversal / bridge behavior. It does not yet include live bitwise predicate evaluation during every graph visit. If C wins on the recall-latency frontier, the next step is to fuse the 216-byte predicate kernel directly into node expansion.


In [ ]:
#@title 1) Experiment settings
FULL_DATA = False #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
M = 24 #@param {type:"integer"}
EF_CONSTRUCTION = 200 #@param {type:"integer"}
SEMANTIC_LAMBDA = 0.35 #@param {type:"number"}
GATE_LOGPROB = -1.0 #@param {type:"number"}
BRIDGE_HOPS = 2 #@param {type:"integer"}
BRIDGE_CAP = 64 #@param {type:"integer"}
POSTFILTER_OVERSAMPLE = 8 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}

print({
    'FULL_DATA': FULL_DATA, 'QUERIES': QUERIES, 'K': K, 'EF': EF,
    'semantic_lambda': SEMANTIC_LAMBDA, 'gate_logprob': GATE_LOGPROB,
    'positive': POSITIVE, 'negative': NEGATIVE,
})


In [ ]:
#@title 2) Clone repo + install Python/Rust dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','faiss-cpu'], check=True)

if shutil.which('rustc') is None or shutil.which('cargo') is None:
    print('Installing minimal stable Rust toolchain...')
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    cargo_bin = str(pathlib.Path.home()/'.cargo'/'bin')
    os.environ['PATH'] = cargo_bin + os.pathsep + os.environ.get('PATH','')

print('commit:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
print('python:', sys.version.split()[0])
print('rustc:', subprocess.check_output(['rustc','--version']).decode().strip())
print('cargo:', subprocess.check_output(['cargo','--version']).decode().strip())


In [ ]:
#@title 3) Export real fashion embeddings + compiled semantic programs
import os, pathlib, subprocess, sys, time
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
ASSETS = pathlib.Path('/content/semantic_hnsw_assets')
if ASSETS.exists(): shutil.rmtree(ASSETS)
t0 = time.time()
subprocess.run([sys.executable,'-m','experiments.export_native_finalists','--config',CFG,'--out-dir',str(ASSETS)], check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('assets:', ASSETS)
print('programs:', sorted(p.name for p in (ASSETS/'sidecar_programs').iterdir() if p.is_dir()))


In [ ]:
#@title 4) Compile the semantic-HNSW Rust executable
import subprocess, os, time
os.chdir('/content/ras')
t0=time.time()
subprocess.run(['cargo','build','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','semantic_hnsw'], check=True)
BIN='/content/ras/rust/semantic_engine/target/release/semantic_hnsw'
print(f'compiled in {time.time()-t0:.1f}s')
print(BIN)


In [ ]:
#@title 5) Run Variant C vs HNSW baselines
import subprocess, pathlib, time
OUT = pathlib.Path('/content/semantic_hnsw_results.csv')
cmd = [
    BIN,
    '--assets', str(ASSETS),
    '--programs', str(ASSETS/'sidecar_programs'),
    '--positive', POSITIVE,
    '--negative', NEGATIVE,
    '--queries', str(QUERIES),
    '--k', str(K),
    '--ef', str(EF),
    '--m', str(M),
    '--ef-construction', str(EF_CONSTRUCTION),
    '--semantic-lambda', str(SEMANTIC_LAMBDA),
    '--gate-logprob', str(GATE_LOGPROB),
    '--bridge-hops', str(BRIDGE_HOPS),
    '--bridge-cap', str(BRIDGE_CAP),
    '--postfilter-oversample', str(POSTFILTER_OVERSAMPLE),
    '--out', str(OUT),
]
print(' '.join(cmd))
t0=time.time()
run = subprocess.run(cmd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'semantic_hnsw failed with code {run.returncode}')
print(f'total wall time: {time.time()-t0:.1f}s')


In [ ]:
#@title 6) Summarize recall / latency frontier
import pandas as pd, numpy as np
df = pd.read_csv('/content/semantic_hnsw_results.csv')
summary = (df.groupby('method')
    .agg(
        queries=('query_id','count'),
        mean_latency_ms=('latency_ms','mean'),
        p50_latency_ms=('latency_ms','median'),
        p95_latency_ms=('latency_ms', lambda x: np.quantile(x, .95)),
        mean_recall_at_k=('recall_at_k','mean'),
        p10_recall_at_k=('recall_at_k', lambda x: np.quantile(x, .10)),
        mean_returned=('returned','mean'),
        mean_visited=('visited','mean'),
        mean_semantic_evals=('semantic_evals','mean'),
        mean_bridge_candidates=('bridge_candidates','mean'),
        qualified_fraction=('qualified_fraction','mean'),
    )
    .reset_index())
display(summary.sort_values('mean_latency_ms'))

pivot_r = df.pivot(index='query_id', columns='method', values='recall_at_k')
pivot_t = df.pivot(index='query_id', columns='method', values='latency_ms')
if 'semantic_hnsw_c' in pivot_r and 'hnsw_filtered' in pivot_r:
    print('C - filtered mean recall delta:', round((pivot_r.semantic_hnsw_c-pivot_r.hnsw_filtered).mean(),4))
    print('C / filtered mean latency ratio:', round((pivot_t.semantic_hnsw_c/pivot_t.hnsw_filtered).mean(),3))
if 'semantic_hnsw_c' in pivot_r and 'hnsw_postfilter' in pivot_r:
    print('C - postfilter mean recall delta:', round((pivot_r.semantic_hnsw_c-pivot_r.hnsw_postfilter).mean(),4))


In [ ]:
#@title 7) Plot latency vs recall
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,5))
for _, r in summary.iterrows():
    ax.scatter(r['mean_latency_ms'], r['mean_recall_at_k'], s=80)
    ax.annotate(r['method'], (r['mean_latency_ms'], r['mean_recall_at_k']), xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Mean query latency (ms)')
ax.set_ylabel(f'Mean Recall@{K}')
ax.set_title('Semantic-aware HNSW recall-latency frontier')
ax.grid(True, alpha=.25)
plt.show()


In [ ]:
#@title 8) Optional small sweep over semantic steering strength
RUN_SWEEP = False #@param {type:"boolean"}
if RUN_SWEEP:
    import pandas as pd, subprocess, pathlib
    rows=[]
    for lam in [0.0, 0.15, 0.35, 0.6]:
        out=pathlib.Path(f'/content/semantic_hnsw_lambda_{lam}.csv')
        sweep_cmd = cmd.copy()
        sweep_cmd[sweep_cmd.index('--semantic-lambda')+1] = str(lam)
        sweep_cmd[sweep_cmd.index('--queries')+1] = str(min(QUERIES,50))
        sweep_cmd[sweep_cmd.index('--out')+1] = str(out)
        subprocess.run(sweep_cmd, check=True)
        z=pd.read_csv(out)
        c=z[z.method=='semantic_hnsw_c']
        rows.append({'lambda':lam,'latency_ms':c.latency_ms.mean(),'recall':c.recall_at_k.mean(),'visited':c.visited.mean(),'bridges':c.bridge_candidates.mean()})
    display(pd.DataFrame(rows))


In [ ]:
#@title 9) Package result artifacts
import pathlib, shutil, json, platform, subprocess
PKG=pathlib.Path('/content/semantic_hnsw_artifact')
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir()
shutil.copy('/content/semantic_hnsw_results.csv', PKG/'semantic_hnsw_results.csv')
summary.to_csv(PKG/'summary.csv', index=False)
meta={
  'commit': subprocess.check_output(['git','-C','/content/ras','rev-parse','HEAD']).decode().strip(),
  'full_data': FULL_DATA, 'queries': QUERIES, 'k': K, 'ef': EF, 'm': M,
  'ef_construction': EF_CONSTRUCTION, 'semantic_lambda': SEMANTIC_LAMBDA,
  'gate_logprob': GATE_LOGPROB, 'bridge_hops': BRIDGE_HOPS, 'bridge_cap': BRIDGE_CAP,
  'postfilter_oversample': POSTFILTER_OVERSAMPLE, 'positive': POSITIVE, 'negative': NEGATIVE,
  'python': platform.python_version(), 'platform': platform.platform(),
  'cpu': pathlib.Path('/proc/cpuinfo').read_text().split('model name')[1].split('\n')[0].split(':',1)[-1].strip() if pathlib.Path('/proc/cpuinfo').exists() and 'model name' in pathlib.Path('/proc/cpuinfo').read_text() else 'unknown',
  'timing_scope': 'HNSW traversal with semantic scores precomputed before query timing; not yet live predicate execution per visited node'
}
(PKG/'environment.json').write_text(json.dumps(meta,indent=2))
shutil.make_archive('/content/rsa_semantic_hnsw_c','zip',PKG)
print('/content/rsa_semantic_hnsw_c.zip')
